# LLM Benchmark — Colab Runner

Model değiştirmek için sadece **1. hücreyi** düzenle, sonra **Run All** yap.

| Bölüm | Açıklama |
|-------|----------|
| 1. Config | Model ve benchmark ayarları (sadece buraya dokun) |
| 2-5. Altyapı | Kurulum, vLLM başlatma, health check |
| 6. Benchmark | Test çalıştırma |
| 7. Sonuçlar | Rapor ve karşılaştırma |
| 8-9. Opsiyonel | ngrok + keepalive (dışarıdan erişim için) |

## 1) Yapılandırma
Sadece bu hücreyi düzenle.

In [ ]:
#  ╔══════════════════════════════════════════════════════════════╗
#  ║  MODEL DEĞİŞTİRMEK İÇİN SADECE BU HÜCREYİ DÜZENLE       ║
#  ╚══════════════════════════════════════════════════════════════╝

# --- Model ---
MODEL_REPO = "Qwen/Qwen3-0.6B"
MODEL_LABEL = "qwen3-0.6b"               # benchmark_data/models.json'daki isim

# --- vLLM parametreleri ---
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.90
VLLM_PORT = 8090
EXTRA_VLLM_ARGS = []                      # modele özel ek argümanlar

# --- Benchmark ---
TEST_SETS = ["coding"]                     # çalıştırılacak test setleri
TEMPERATURE = 0.7
MAX_TOKENS = 1024
TOP_P = 1.0

# --- Opsiyonel: ngrok (dışarıdan erişim için) ---
NGROK_ENABLED = False
NGROK_AUTHTOKEN = "BURAYA_NGROK_TOKENINI_YAZ"

# --- Türetilen değerler (dokunma) ---
VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}"
print(f"Model: {MODEL_REPO}")
print(f"Test setleri: {TEST_SETS}")
print(f"vLLM: {VLLM_BASE_URL}")

## 2) Kurulum

In [ ]:
!pip -q install -U vllm pyngrok httpx pydantic hypothesis nest_asyncio
!nvidia-smi

## 3) Proje klonu ve dizinler

In [ ]:
import os, sys

ROOT = "/content/llm"
LOG_DIR = f"{ROOT}/logs"
CACHE_DIR = f"{ROOT}/cache"
PROJECT_DIR = f"{ROOT}/benchmark"

for p in [ROOT, LOG_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_DIR

# Git clone veya pull
REPO_URL = "https://github.com/orhan-kaplan/benchmark.git"

if os.path.exists(PROJECT_DIR):
    print("📦 Repo mevcut, güncelleniyor...")
    !git -C {PROJECT_DIR} pull --ff-only
else:
    print("📦 Repo klonlanıyor...")
    !git clone {REPO_URL} {PROJECT_DIR}

# Import path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Doğrulama
pkg_ok = os.path.exists(os.path.join(PROJECT_DIR, 'benchmark', '__init__.py'))
data_ok = os.path.exists(os.path.join(PROJECT_DIR, 'benchmark_data', 'models.json'))
print(f"\nProje: {PROJECT_DIR}")
print(f"benchmark/: {'✅' if pkg_ok else '❌'}")
print(f"benchmark_data/: {'✅' if data_ok else '❌'}")
if not pkg_ok or not data_ok:
    raise FileNotFoundError('Repo yapısı hatalı')

## 4) vLLM server başlat ve bekle

In [ ]:
import subprocess, time, requests, os
from IPython.display import clear_output

LOG_PATH = f"{LOG_DIR}/vllm-{MODEL_LABEL}.log"

# --- Önceki server'ı durdur ---
subprocess.run("pkill -f 'vllm serve' || true", shell=True, check=False)
time.sleep(2)

# --- vLLM'i başlat ---
cmd = [
    "vllm", "serve", MODEL_REPO,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--trust-remote-code",
    *EXTRA_VLLM_ARGS,
]

with open(LOG_PATH, "w") as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
    )

print(f"🚀 vLLM başlatıldı (PID: {proc.pid})")
print(f"   Model: {MODEL_REPO}")
print(f"   Port: {VLLM_PORT}")
print(f"   Log: {LOG_PATH}")
print()

# --- Canlı log takibi ile bekle ---
def get_log_tail(n=8):
    try:
        with open(LOG_PATH, 'r') as f:
            lines = f.readlines()
            return ''.join(lines[-n:])
    except:
        return ''

def detect_phase(log_text):
    lower = log_text.lower()
    if 'error' in lower or 'exception' in lower or 'traceback' in lower:
        return '❌ HATA'
    if 'downloading' in lower or 'fetching' in lower:
        return '📥 Model indiriliyor'
    if 'loading model' in lower or 'loading weights' in lower:
        return '🔄 Model yükleniyor (GPU\'a aktarılıyor)'
    if 'warming up' in lower or 'cuda graph' in lower:
        return '🔥 CUDA warmup'
    if 'started server' in lower or 'uvicorn running' in lower:
        return '✅ Server hazır'
    return '⏳ Başlatılıyor'

server_ready = False
start_time = time.time()
timeout = 600

while time.time() - start_time < timeout:
    elapsed = int(time.time() - start_time)
    log_tail = get_log_tail(8)
    phase = detect_phase(log_tail)

    # Health check
    try:
        r = requests.get(f"{VLLM_BASE_URL}/health", timeout=3)
        if r.status_code == 200:
            server_ready = True
            break
    except:
        pass

    # Process crash kontrolü
    if proc.poll() is not None:
        print(f"❌ vLLM çöktü! (exit code: {proc.returncode})")
        print(f"\nSon log satırları:")
        print(get_log_tail(20))
        break

    # Durum göster
    print(f"[{elapsed:3d}s] {phase}")
    if '❌' in phase:
        print(f"\nSon log satırları:")
        print(get_log_tail(15))
        break

    time.sleep(5)

if server_ready:
    elapsed = int(time.time() - start_time)
    print(f"\n✅ Server hazır! ({elapsed}s)")
    models = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=5).json()
    for m in models.get('data', []):
        print(f"   Model: {m['id']}")

    # Hızlı test
    r = requests.post(
        f"{VLLM_BASE_URL}/v1/chat/completions",
        json={'model': MODEL_REPO, 'messages': [{'role': 'user', 'content': 'Say hello in one sentence.'}],
              'temperature': 0.7, 'max_tokens': 32},
        timeout=60,
    )
    if r.status_code == 200:
        print(f"   💬 Test: {r.json()['choices'][0]['message']['content'][:150]}")
    else:
        print(f"   ⚠️ Test başarısız: {r.status_code}")
elif not server_ready:
    print(f"\n❌ Timeout ({timeout}s)! Son log:")
    print(get_log_tail(20))

def stop_vllm():
    subprocess.run('pkill -f "vllm serve" || true', shell=True, check=False)
    time.sleep(2)
    print('Server durduruldu.')

## 6) Benchmark çalıştır

In [ ]:
import asyncio
from pathlib import Path

from benchmark.runner import BenchmarkRunner
from benchmark.catalog import ModelCatalog
from benchmark.test_sets import TestSetManager
from benchmark.api_client import APIClient
from benchmark.metrics import MetricsCollector, VRAMTracker
from benchmark.storage import StorageManager
from benchmark.models import BenchmarkRunConfig, GenerationParams, ModelEntry, Backend

data_path = Path(PROJECT_DIR) / "benchmark_data"

# Bileşenleri oluştur
storage = StorageManager(data_path)
catalog = ModelCatalog(data_path)
test_sets = TestSetManager(data_path)
api_client = APIClient(timeout=180.0, max_retries=2)
metrics = MetricsCollector()
vram_tracker = VRAMTracker()

# Modeli katalogda güncelle (endpoint'i localhost olarak ayarla)
model_entry = catalog.get(MODEL_LABEL)
if model_entry is None:
    model_entry = ModelEntry(
        name=MODEL_LABEL,
        repo=MODEL_REPO,
        format="HF",
        backend=Backend.VLLM,
        tags=[],
        api_endpoint=VLLM_BASE_URL,
    )
    catalog.register(model_entry)
    print(f"Model kataloga eklendi: {MODEL_LABEL}")
else:
    catalog.update(MODEL_LABEL, {"api_endpoint": VLLM_BASE_URL})
    print(f"Model endpoint güncellendi: {MODEL_LABEL} → {VLLM_BASE_URL}")

# Runner oluştur
runner = BenchmarkRunner(
    catalog=catalog,
    test_set_manager=test_sets,
    api_client=api_client,
    metrics_collector=metrics,
    storage=storage,
    vram_tracker=vram_tracker,
)

# Async benchmark fonksiyonu
async def run_benchmarks():
    run_ids = []
    for test_set_name in TEST_SETS:
        print(f"\n{'='*60}")
        print(f"Benchmark: {test_set_name} × {MODEL_LABEL}")
        print(f"{'='*60}")

        config = BenchmarkRunConfig(
            test_set_name=test_set_name,
            model_names=[MODEL_LABEL],
            params=GenerationParams(
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                top_p=TOP_P,
            ),
        )

        result = await runner.run(config)
        run_ids.append(result.run_id)

        success = sum(1 for r in result.results if r.success)
        failed = sum(1 for r in result.results if not r.success)
        print(f"\n✅ Tamamlandı: run_id={result.run_id}")
        print(f"   Başarılı: {success}, Başarısız: {failed}")

    await api_client.close()
    return run_ids

# Çalıştır (Colab ve standart Python uyumlu)
try:
    loop = asyncio.get_running_loop()
    import nest_asyncio
    nest_asyncio.apply()
    run_ids = asyncio.run(run_benchmarks())
except RuntimeError:
    run_ids = asyncio.run(run_benchmarks())

print(f"\n🏁 Tüm benchmark'lar tamamlandı. Run ID'ler: {run_ids}")

## 7) Sonuçlar ve rapor

In [ ]:
from benchmark.reporter import ReportGenerator

reporter = ReportGenerator(storage=storage)

for run_id in run_ids:
    print(f"\n{'='*60}")
    print(f"Rapor: {run_id}")
    print(f"{'='*60}")

    # Özet
    summary = reporter.generate_summary(run_id)
    print(f"  Toplam prompt: {summary.total_prompts}")
    print(f"  Toplam model: {summary.total_models}")
    print(f"  Başarılı: {summary.successful_results}")
    print(f"  Başarısız: {summary.failed_results}")

    # Sonuçları göster
    results = storage.read_results(run_id)
    print(f"\n  Detaylı sonuçlar:")
    for r in results:
        status = "✅" if r.get("success") else "❌"
        m = r.get("metrics") or {}
        time_ms = m.get("total_time_ms") or 0
        tps = m.get("tokens_per_second") or 0
        ttft = m.get("ttft_ms")
        ttft_str = f"{ttft:.0f}ms" if ttft else "-"
        comp_tokens = m.get("completion_tokens") or 0

        print(f"  {status} {r['prompt_id']:15s} | {time_ms:8.0f}ms | {tps:5.1f} tok/s | TTFT: {ttft_str:>7s} | {comp_tokens} tokens")

        if r.get("error"):
            print(f"     ⚠️ Hata: {r['error'][:100]}")

    # VRAM raporu
    vram_report = reporter.generate_vram_report(run_id, vram_limit_mb=16384)
    if vram_report.models_within_limit or vram_report.models_exceeding_limit:
        print(f"\n  VRAM (16GB limit):")
        for m in vram_report.models_within_limit:
            print(f"    ✅ {m['name']}: peak={m['peak_mb']:.0f}MB, avg={m['avg_mb']:.0f}MB")
        for m in vram_report.models_exceeding_limit:
            print(f"    ❌ {m['name']}: peak={m['peak_mb']:.0f}MB, avg={m['avg_mb']:.0f}MB")

    # JSON rapor kaydet
    report_path = storage.runs_dir / run_id / "report.json"
    reporter.export_json(run_id, report_path)
    print(f"\n  📄 Rapor: {report_path}")

### Yanıtları incele

In [ ]:
# Belirli bir prompt'un yanıtını görmek için:
# run_id ve prompt_id'yi değiştir

INSPECT_RUN = run_ids[0] if run_ids else ""
INSPECT_PROMPT = None  # None = hepsini göster, veya "code-001" gibi belirli bir ID

if INSPECT_RUN:
    results = storage.read_results(INSPECT_RUN)
    for r in results:
        if INSPECT_PROMPT and r["prompt_id"] != INSPECT_PROMPT:
            continue
        print(f"\n{'─'*60}")
        print(f"Prompt: {r['prompt_id']} | Model: {r['model_name']}")
        print(f"{'─'*60}")
        if r.get("response_text"):
            print(r["response_text"][:2000])
        elif r.get("error"):
            print(f"HATA: {r['error']}")

### Sonuçları indir

In [ ]:
import shutil, os
from google.colab import files

if run_ids:
    # Her run için zip oluştur ve indir
    for run_id in run_ids:
        run_dir = str(storage.runs_dir / run_id)
        zip_path = f"/content/{run_id}"

        # CSV rapor da oluştur
        csv_path = storage.runs_dir / run_id / "report.csv"
        reporter.export_csv(run_id, csv_path)

        # Zip
        shutil.make_archive(zip_path, 'zip', run_dir)
        print(f"📦 {run_id}.zip oluşturuldu")
        print(f"   İçerik: meta.json, results.jsonl, report.json, report.csv")

        # İndir
        files.download(f"{zip_path}.zip")
else:
    print("İndirilecek sonuç yok. Önce benchmark çalıştır.")

## 8) Opsiyonel: ngrok tunnel (dışarıdan erişim)

In [ ]:
PUBLIC_URL = VLLM_BASE_URL  # varsayılan: localhost

if NGROK_ENABLED:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    ngrok.kill()
    tunnel = ngrok.connect(VLLM_PORT, "http")
    PUBLIC_URL = tunnel.public_url
    print(f"🌐 Public URL: {PUBLIC_URL}")
    print(f"Open WebUI → API Base URL: {PUBLIC_URL}/v1")
else:
    print("ngrok devre dışı. Etkinleştirmek için config'de NGROK_ENABLED = True yap.")

## 9) Opsiyonel: Canlı tut

In [ ]:
import time, requests

print(f"Server: {VLLM_BASE_URL}")
print("Durdurmak için interrupt et.\n")

while True:
    try:
        r = requests.get(f"{VLLM_BASE_URL}/health", timeout=5)
        s = "✅" if r.status_code == 200 else "⚠️"
    except:
        s = "❌"
    print(f"{s} {time.strftime('%H:%M:%S')}")
    time.sleep(30)

---
### Yardımcı komutlar
```python
# Log kontrol
!cat /content/llm/logs/vllm-*.log | tail -50

# GPU kullanımı
!nvidia-smi

# Server durdur
stop_vllm()

# Mevcut benchmark çalıştırmalarını listele
storage.list_runs()
```